In [2]:
# ==============================================================
# Ex. No: 6
# RETRIEVAL-AUGMENTED GENERATION (RAG)
# USING VECTOR DATABASES
# ==============================================================

# Install required libraries
!pip -q install sentence-transformers faiss-cpu transformers accelerate torch

import faiss
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ==============================================================
# STEP 1 : KNOWLEDGE BASE
# ==============================================================

documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]

print("=" * 60)
print("KNOWLEDGE BASE")
print("=" * 60)

for i, doc in enumerate(documents, 1):
    print(f"{i}. {doc}")

# ==============================================================
# STEP 2 : CREATE EMBEDDINGS
# ==============================================================

print("\nLoading Sentence Transformer Model...")

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embed_model.encode(
    documents,
    convert_to_numpy=True
)

print("Document Embeddings Shape:", doc_embeddings.shape)

# ==============================================================
# STEP 3 : CREATE FAISS VECTOR DATABASE
# ==============================================================

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(doc_embeddings.astype("float32"))

print("Number of documents indexed:", index.ntotal)

# ==============================================================
# STEP 4 : USER QUERY
# ==============================================================

query = "What is RAG in AI?"

print("\nUser Query:")
print(query)

query_embedding = embed_model.encode(
    [query],
    convert_to_numpy=True
)

distances, indices = index.search(
    query_embedding.astype("float32"),
    k=2
)

retrieved_chunks = [documents[i] for i in indices[0]]

print("\nRetrieved Context:")

for chunk in retrieved_chunks:
    print("-", chunk)

# ==============================================================
# STEP 5 : BUILD PROMPT
# ==============================================================

context = "\n".join(retrieved_chunks)

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

# ==============================================================
# STEP 6 : LOAD FLAN-T5 MODEL
# ==============================================================

print("\nLoading FLAN-T5 Model...")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)

# ==============================================================
# STEP 7 : GENERATE ANSWER
# ==============================================================

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

# ==============================================================
# STEP 8 : DISPLAY RESULT
# ==============================================================

print("\n" + "=" * 60)
print("GENERATED ANSWER")
print("=" * 60)

print(answer)

print("\n" + "=" * 60)
print("EXPERIMENT COMPLETED SUCCESSFULLY")
print("=" * 60)

KNOWLEDGE BASE
1. The Eiffel Tower is located in Paris, France and was completed in 1889.
2. Retrieval-Augmented Generation combines document retrieval with text generation.
3. Python is a popular high-level programming language used in AI development.
4. Vector databases store embeddings and support fast similarity search.

Loading Sentence Transformer Model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Document Embeddings Shape: (4, 384)
Number of documents indexed: 4

User Query:
What is RAG in AI?

Retrieved Context:
- Python is a popular high-level programming language used in AI development.
- Retrieval-Augmented Generation combines document retrieval with text generation.

Loading FLAN-T5 Model...


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


GENERATED ANSWER
combines document retrieval with text generation

EXPERIMENT COMPLETED SUCCESSFULLY
